<h3 align='center'>AIRLINE</h3>
<h3 align='center'>AIRLINE OPERATIONS, DELAY & RELIABILITY ANALYTICS</h3>

<h4 align='center'>NOTEBOOK 06 - FEATURE ENGINEERING</h4>

**Objective:** Create business-oriented analytical features from the validated and transformed dataset.

 **Important:** Feature must have a clear analytical purpose. Do not create derived columns merely to increase column count.

## Import Paths

In [18]:
from pathlib import Path
import pandas as pd
import numpy as np

## Project Paths

In [19]:
PROJECT_PATH = Path.cwd().parent

CLEANED_PATH = PROJECT_PATH / "04_Cleaned_Data"

TRANSFORMED_FILE = (
CLEANED_PATH /
"Airline_Flights_January_2024_Cleaned.csv"
)

In [20]:
df = pd.read_csv(
    TRANSFORMED_FILE,
    low_memory=False
)

df["FlightDate"] = pd.to_datetime(
    df["FlightDate"],
    errors="coerce"
)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

Rows: 547,271
Columns: 93


## Feature Engineering Inventory

In [21]:
feature_plan = pd.DataFrame({
    "Feature": [
        "Flight_Year",
        "Flight_Month",
        "Flight_Day",
        "Flight_Day_Name",
        "Route"
    ],

    "Business_Purpose": [
        "Year-level comparison",
        "Period-level analysis",
        "Daily operational analysis",
        "Day-of-week analysis",
        "Route-level reliability analysis"
    ]
})

display(feature_plan)

,Feature,Business_Purpose
0,Flight_Year,Year-level comparison
1,Flight_Month,Period-level analysis
2,Flight_Day,Daily operational analysis
3,Flight_Day_Name,Day-of-week analysis
4,Route,Route-level reliability analysis


## Cancellation Status

In [22]:
if "Cancelled" in df.columns:

    df["Cancellation_Status"] = np.where(
    df["Cancelled"] == 1,
    "Cancelled",
    "Completed"
    )

print(
    df["Cancellation_Status"]
    .value_counts(dropna=False)
)

Cancellation_Status
Completed    526882
Cancelled     20389
Name: count, dtype: int64


## Diversion Status

In [23]:
if "Diverted" in df.columns:

    df["Diversion_Status"] = np.where(
        df["Diverted"] == 1,
        "Diverted",
        "Not Diverted"
    )

    print(
    df["Diversion_Status"]
    .value_counts(dropna=False)
    )

Diversion_Status
Not Diverted    545759
Diverted          1512
Name: count, dtype: int64


## Departure Delay Status

In [25]:
if "DepDel15" in df.columns:

    df["Departure_Delay_Status"] = np.select(
        [
            df["DepDel15"] == 1,
            df["DepDel15"] == 0
        ],
        [
            "15+ Minute Delay",
            "Less Than 15 Minute Delay"
        ],
        default="Unknown"
    )

    print(
        df["Departure_Delay_Status"]
        .value_counts(dropna=False)
    )

Departure_Delay_Status
Less Than 15 Minute Delay    405154
15+ Minute Delay             122259
Unknown                       19858
Name: count, dtype: int64


## Arrival Delay Status

In [26]:
if "ArrDel15" in df.columns:

    df["Arrival_Delay_Status"] = np.select(
        [
            df["ArrDel15"] == 1,
            df["ArrDel15"] == 0
        ],
        [
            "15+ Minute Delay",
            "Less Than 15 Minute Delay"
        ],
        default="Unknown"
    )

    print(
    df["Arrival_Delay_Status"]
    .value_counts(dropna=False)
    )

Arrival_Delay_Status
Less Than 15 Minute Delay    398960
15+ Minute Delay             126410
Unknown                       21901
Name: count, dtype: int64


## Delay Magnitude Classification

In [27]:
if "ArrDelayMinutes" in df.columns:

    df["Arrival_Delay_Band"] = pd.cut(
        df["ArrDelayMinutes"],
        bins=[
            -np.inf,
            0,
            15,
            30,
            60,
            120,
            np.inf
        ],
        labels=[
            "Early / On Time",
            "0-15 Minutes",
            "15-30 Minutes",
            "30-60 Minutes",
            "60-120 Minutes",
            "120+ Minutes"
        ]
    )

    display(
        df["Arrival_Delay_Band"]
        .value_counts(dropna=False)
        .sort_index()
    )

Arrival_Delay_Band
Early / On Time    315970
0-15 Minutes        86892
15-30 Minutes       40696
30-60 Minutes       36422
60-120 Minutes      25881
120+ Minutes        19509
NaN                 21901
Name: count, dtype: int64

## Arrival Delay Band

In [28]:
if "ArrDelayMinutes" in df.columns:

    df["Arrival_Delay_Band"] = pd.cut(
        df["ArrDelayMinutes"],
        bins=[
            -np.inf,
            0,
            15,
            30,
            60,
            120,
            np.inf
        ],      
        labels=[
            "Early / On Time",
            "0-15 Minutes",
            "15-30 Minutes",
            "30-60 Minutes",
            "60-120 Minutes",
            "120+ Minutes"
        ]
    )
    display(
        df["Arrival_Delay_Band"]
        .value_counts(dropna=False)
        .sort_index()
    )

Arrival_Delay_Band
Early / On Time    315970
0-15 Minutes        86892
15-30 Minutes       40696
30-60 Minutes       36422
60-120 Minutes      25881
120+ Minutes        19509
NaN                 21901
Name: count, dtype: int64

## Departure Delay Band

In [29]:
if "DepDelayMinutes" in df.columns:

    df["Departure_Delay_Band"] = pd.cut(
        df["DepDelayMinutes"],
        bins=[
            -np.inf,
            0,
            15,
            30,
            60,
            120,
            np.inf
        ],
        labels=[
            "Early / On Time",
            "0-15 Minutes",
            "15-30 Minutes",
            "30-60 Minutes",
            "60-120 Minutes",
            "120+ Minutes"
        ]
    )

## Delay Cause Total

In [30]:
delay_cause_columns = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

available_delay_causes = [
    column
    for column in delay_cause_columns
    if column in df.columns
]

print(
"Available delay-cause columns:",
available_delay_causes
)

Available delay-cause columns: ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']


In [31]:
if available_delay_causes:

    df["Total_Reported_Delay_Cause_Minutes"] = (
        df[available_delay_causes]
        .fillna(0)
        .sum(axis=1)
    )

    print(
        "Total reported delay-cause feature created."
    )
else:

    print(
        "No delay-cause columns available; "
        "feature not created."
    )

Total reported delay-cause feature created.


## Route-Level Feature Preparation

In [32]:
if "Route" in df.columns:

    route_counts = (
        df["Route"]
        . value_counts()
        .rename_axis("Route")
        .reset_index(name="Flight_Count")
    )

    display(route_counts.head(20))

## Feature Inventory

In [33]:
derived_columns = [
    "Flight_Year",
    "Flight_Month",
    "Flight_Day",
    "Flight_Day_Name",
    "Route",
    "Cancellation_Status",
    "Diversion_Status",
    "Departure_Delay_Status",
    "Arrival_Delay_Status",
    "Arrival_Delay_Band",
    "Departure_Delay_Band",
    "Total_Reported_Delay_Cause_Minutes"
]

derived_columns = [
    column
    for column in derived_columns
    if column in df.columns
]

feature_inventory = pd.DataFrame({
    "Derived_Feature": derived_columns,
    "Purpose": [
        "Analytical representation / business segmentation"
        for _ in derived_columns
    ]
})

display(feature_inventory)

,Derived_Feature,Purpose
0,Cancellation_Status,Analytical representation / business segmentation
1,Diversion_Status,Analytical representation / business segmentation
2,Departure_Delay_Status,Analytical representation / business segmentation
3,Arrival_Delay_Status,Analytical representation / business segmentation
4,Arrival_Delay_Band,Analytical representation / business segmentation
5,Departure_Delay_Band,Analytical representation / business segmentation
6,Total_Reported_Delay_Cause_Minutes,Analytical representation / business segmentation


## Feature Validation

In [35]:
print("=" * 65)
print("FEATURE ENGINEERING VALIDATION")
print("=" * 65)

print("Rows:", f"{len(df):,}")

for column in derived_columns:
    print(
        f"{column:<40} : "
        f"{df[column].notna().sum():,} populated"
    )

FEATURE ENGINEERING VALIDATION
Rows: 547,271
Cancellation_Status                      : 547,271 populated
Diversion_Status                         : 547,271 populated
Departure_Delay_Status                   : 547,271 populated
Arrival_Delay_Status                     : 547,271 populated
Arrival_Delay_Band                       : 525,370 populated
Departure_Delay_Band                     : 527,413 populated
Total_Reported_Delay_Cause_Minutes       : 547,271 populated


## Save Feature Dataset

In [36]:
FEATURED_FILE = (
    CLEANED_PATH /
    "Airline_Flights_Feature_Engineered. csv"
)

df.to_csv(
    FEATURED_FILE,
    index=False
)

print("Feature-engineered dataset saved:")
print(FEATURED_FILE. resolve())

Feature-engineered dataset saved:
D:\arc\Python\Python Projects\airline_operations_analytics\04_Cleaned_Data\Airline_Flights_Feature_Engineered. csv


## Feature Engineering Audit

In [37]:
feature_audit = pd.DataFrame({
    "Feature": derived_columns,
    "Type": [
        "Derived analytical feature"
        for _ in derived_columns
    ],
    "Status": [
        "CREATED"
        for _ in derived_columns
    ]
})

audit_file = (
    PROJECT_PATH /
    "02_Documentation" /
    "05_Feature_Engineering_Audit.xlsx"
)

feature_audit.to_excel(
    audit_file,
    index=False
)

print("Feature audit saved:")
print(audit_file.resolve())

Feature audit saved:
D:\arc\Python\Python Projects\airline_operations_analytics\02_Documentation\05_Feature_Engineering_Audit.xlsx
